In [2]:
import datasets, subprocess
from ai_metaphors.utils.text_utils import extract_json
from ai_metaphors.providers.grazie_provider import GrazieProvider
from ai_metaphors.providers.manim_provider import ManimProvider
from grazie.api.client.gateway import AuthType, GrazieApiGatewayClient, GrazieAgent
from grazie.api.client.endpoints import GrazieApiGatewayUrls

In [6]:
with open("/home/ynoviello/PycharmProjects/AI_Metaphors/token.secret", 'r') as t: token = t.read()

client = GrazieApiGatewayClient(
    grazie_agent=GrazieAgent(name="grazie-api-gateway-client-readme", version="dev"),
    url=GrazieApiGatewayUrls.STAGING,
    grazie_jwt_token=token,
    auth_type=AuthType.USER,
)

In [8]:
provider = GrazieProvider(client, model="openai-gpt-4o")
ds = datasets.load_from_disk("resources/subset")

In [31]:
import re
def extract_content(input_string):
    # Regex pattern to match content inside triple backticks
    print(input_string)
    pattern = r'```(.*?)```'
    match = re.search(pattern, input_string, re.DOTALL)

    # If a match is found, return the content inside backticks
    if match:
        return match.group(1)
    # If no match, return the whole string
    return input_string.strip()


extract_content("```ciao\n```")

```ciao
```


'ciao\n'

In [9]:
ds

Dataset({
    features: ['value', 'definition', 'metaphor'],
    num_rows: 14
})

In [10]:
term = ds[1]
metaphor = term["metaphor"]
term

{'value': 'append',
 'definition': 'The append function is used to update a StringBuilder with new text.',
 'metaphor': 'Imagine you are a scrapbook enthusiast. You have a special scrapbook (StringBuilder) where you collect and paste various memorable photos and notes (text). Each time you experience something new and exciting, you take a new photo or write a new note (new text). \n\nThe append function is like your glue stick. Whenever you want to add a new photo or note to your scrapbook, you use the glue stick to attach it to the next available page. Over time, your scrapbook grows thicker and more filled with memories, just as the StringBuilder grows longer with each new piece of text you append.'}

In [11]:
classes = provider.get_classes(term, metaphor)

In [12]:
classes_dict = extract_json(classes)

In [13]:
# Static analysis with pylint for each element
for i in range(len(classes_dict['elements'])):
    open("/tmp/dummy.py", "w").write(
        f"{classes_dict['elements'][i]['code']}"
    )
    command = ["pylint", "-E", "/tmp/dummy.py"]
    process = subprocess.run(command, capture_output=True, text=True)
    static_errors = process.stdout
    print(f"{i}: {static_errors}")

0: 
1: 
2: 


In [14]:
desc = provider.get_description(term, metaphor, str(classes_dict))
# desc = ""
manim_code = provider.get_manim(term, metaphor, str(classes_dict), desc)

In [19]:
manim_provider = ManimProvider(provider, term,
                               executable="/home/ynoviello/anaconda3/envs/jetbrains/bin",
                               working_dir="manim_stuff")

In [23]:
!ls

__init__.py  notebook.ipynb  providers	  resources
manim_stuff  prompts	     __pycache__  utils


In [26]:
manim_provider.write_python(manim_code, font_path="'./resources/JetBrainsSans-Regular.ttf'",)
error = manim_provider.execute_manim_script()
print(error)

100


In [51]:
# import json
# with open("/home/ynoviello/PycharmProjects/AI_Metaphors/manim_stuff/scripts/best-scripts/replace.json", 'w') as j:
#     json.dump(classes_dict, j)

In [11]:
# Only if needed
# error = manim_provider.fix_code(error)
# print(error)